Imports libraries and defines the theme for the table

In [ ]:
%pip install qdrant-client sentence-transformers rich
%pip install rich-theme-manager

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

# --- STEP 2: VISUAL SETUP ---
from rich.style import Style
from rich_theme_manager import Theme, ThemeManager
import pathlib
import pandas as pd
import warnings
import sys

# --- STEP 4: SETUP VECTOR DB ---
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# Define Theme
THEMES = [
    Theme(
        name="dark",
        description="Dark mode theme",
        tags=["dark"],
        styles={
            "repr.own": Style(color="#e87d3e", bold=True),
            "repr.tag_name": "dim cyan",
            "repr.call": "bright_yellow",
            "repr.str": "bright_green",
            "repr.number": "bright_red",
            "repr.none": "dim white",
            "repr.attrib_name": Style(color="#e87d3e", bold=True),
            "repr.attrib_value": "bright_blue",
            "default": "bright_white on black"
        },
    )
]
theme_dir = pathlib.Path("themes").expanduser()
theme_dir.mkdir(parents=True, exist_ok=True)
theme_manager = ThemeManager(theme_dir=theme_dir, themes=THEMES)
console = Console(theme=theme_manager.get("dark"))

Initial Setup and Loads Dataset

In [ ]:
# --- SETUP (Fast Reload) ---
console = Console()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen2.5-3B-Instruct"

# --- STEP 3: LOAD DATA ---
warnings.filterwarnings('ignore')
console.print("[bold green]Loading NIST Data...[/bold green]")

try:
    df = pd.read_csv('/kaggle/input/datasets/colepaulik/nist-800-53/NIST_SP-800-53.csv')
    df['combined_text'] = (
        "Control ID: " + df['identifier'].astype(str) + "; " +
        "Title: " + df['name'].astype(str) + "; " +
        "Control Text: " + df['control_text'].fillna('').astype(str) + "; " +
        "Discussion: " + df['discussion'].fillna('').astype(str)
    )
    data = df.to_dict('records')
    console.print(f"[dim]Loaded {len(data)} rows successfully.[/dim]")
except FileNotFoundError:
    console.print("[bold red]CRITICAL ERROR: CSV file not found![/bold red]")
    console.print("[yellow]Please re-upload 'NIST_SP-800-53.csv' to the Files panel.[/yellow]")
    data = []

# 1. Load Resources (Only if not already loaded to save time)
if 'qdrant' not in globals():
    console.print("[bold yellow]Reloading Database connection...[/bold yellow]")
    qdrant = QdrantClient(":memory:")
    encoder = SentenceTransformer('all-MiniLM-L6-v2')

    # Index NIST if data exists
    if 'data' in globals() and data:
        qdrant.recreate_collection(
            collection_name="nist_controls",
            vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
        )
        qdrant.upload_points(
            collection_name="nist_controls",
            points=[
                models.PointStruct(id=idx, vector=encoder.encode(doc["combined_text"]).tolist(), payload=doc)
                for idx, doc in enumerate(data)
            ]
        )
        console.print("[bold green]NIST collection indexed.[/bold green]")

    # --- LOAD AND INDEX MITRE D3FEND (separate collection) ---
    console.print("[bold green]Loading MITRE D3FEND Data...[/bold green]")
    try:
        d3fend_path = '/kaggle/input/datasets/colepaulik/mitre-d3fend/mitre_d3fend.csv'  # adjust if needed
        df_d3fend = pd.read_csv(d3fend_path)
        
        # Create combined_text – adapt if column names differ slightly
        df_d3fend['combined_text'] = (
            "D3FEND ID: " + df_d3fend['ID'].astype(str) + "; " +
            "Tactic: " + df_d3fend.get('D3FEND Tactic', '').astype(str) + "; " +
            "Technique: " + df_d3fend.get('D3FEND Technique', '').astype(str) + "; " +
            "Level 0: " + df_d3fend.get('D3FEND Technique Level 0', '').astype(str) + "; " +
            "Level 1: " + df_d3fend.get('D3FEND Technique Level 1', '').astype(str) + "; " +
            "Definition: " + df_d3fend['Definition'].fillna('').astype(str)
        )
        
        d3fend_data = df_d3fend.to_dict('records')
        console.print(f"[dim]Loaded {len(d3fend_data)} D3FEND techniques successfully.[/dim]")
        
        # Create separate collection for D3FEND
        qdrant.recreate_collection(
            collection_name="d3fend_techniques",
            vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
        )
        
        qdrant.upload_points(
            collection_name="d3fend_techniques",
            points=[
                models.PointStruct(id=idx, vector=encoder.encode(doc["combined_text"]).tolist(), payload=doc)
                for idx, doc in enumerate(d3fend_data)
            ]
        )
        console.print("[bold green]D3FEND collection indexed.[/bold green]")
    
    except FileNotFoundError:
        console.print("[bold red]D3FEND CSV not found! Path checked: " + d3fend_path + "[/bold red]")
        d3fend_data = []

    # --- LOAD AND INDEX CISA KEV (Known Exploited Vulnerabilities) ---
    console.print("[bold green]Loading CISA KEV Data...[/bold green]")
    try:
        kev_path = '/kaggle/input/datasets/colepaulik/known-ev/known_exploited_vulnerabilities.csv'  # Adjust path if needed
        df_kev = pd.read_csv(kev_path)
        
        # Construct combined_text for KEV
        df_kev['combined_text'] = (
            "CVE ID: " + df_kev['cveID'].astype(str) + "; " +
            "Vendor: " + df_kev['vendorProject'].astype(str) + "; " +
            "Product: " + df_kev['product'].astype(str) + "; " +
            "Vulnerability: " + df_kev['vulnerabilityName'].astype(str) + "; " +
            "Description: " + df_kev['shortDescription'].fillna('').astype(str) + "; " +
            "Required Action: " + df_kev['requiredAction'].fillna('').astype(str) + "; " +
            "CWEs: " + df_kev['cwes'].fillna('').astype(str)
        )
        
        kev_data = df_kev.to_dict('records')
        console.print(f"[dim]Loaded {len(kev_data)} KEV records successfully.[/dim]")
        
        # Create collection for KEV
        qdrant.recreate_collection(
            collection_name="known_exploited_vulnerabilities",
            vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
        )
        
        qdrant.upload_points(
            collection_name="known_exploited_vulnerabilities",
            points=[
                models.PointStruct(id=idx, vector=encoder.encode(doc["combined_text"]).tolist(), payload=doc)
                for idx, doc in enumerate(kev_data)
            ]
        )
        console.print("[bold green]CISA KEV collection indexed.[/bold green]")
        
    except FileNotFoundError:
        console.print(f"[bold red]KEV CSV not found! Path checked: {kev_path}[/bold red]")
        kev_data = []

    # --- LOAD AND INDEX MITRE ENTERPRISE ATT&CK ---
    console.print("[bold green]Loading MITRE Enterprise ATT&CK Data...[/bold green]")
    try:
        mitre_path = '/kaggle/input/datasets/colepaulik/mitre-attack/mitre-enterprise-attack-v18.1.csv'  # Adjust path if needed
        df_mitre = pd.read_csv(mitre_path)
        
        # Construct combined_text for MITRE ATT&CK
        # We focus on ID, Name, Tactics, and Description for better retrieval
        df_mitre['combined_text'] = (
            "MITRE ID: " + df_mitre['ID'].astype(str) + "; " +
            "Technique Name: " + df_mitre['name'].astype(str) + "; " +
            "Tactics: " + df_mitre['tactics'].fillna('').astype(str) + "; " +
            "Platforms: " + df_mitre['platforms'].fillna('').astype(str) + "; " +
            "Description: " + df_mitre['description'].fillna('').astype(str)
        )
        
        mitre_data = df_mitre.to_dict('records')
        console.print(f"[dim]Loaded {len(mitre_data)} MITRE techniques successfully.[/dim]")
        
        # Create collection for MITRE ATT&CK
        qdrant.recreate_collection(
            collection_name="mitre_attack",
            vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
        )
        
        qdrant.upload_points(
            collection_name="mitre_attack",
            points=[
                models.PointStruct(id=idx, vector=encoder.encode(doc["combined_text"]).tolist(), payload=doc)
                for idx, doc in enumerate(mitre_data)
            ]
        )
        console.print("[bold green]MITRE ATT&CK collection indexed.[/bold green]")
        
    except FileNotFoundError:
        console.print(f"[bold red]MITRE CSV not found! Path checked: {mitre_path}[/bold red]")
        mitre_data = []

if 'model' not in globals():
    console.print(f"[bold yellow]Loading Model ({device})...[/bold yellow]")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)

Chat Loop

In [ ]:
# UI without the persistent memory

'''
import gradio as gr
from threading import Thread
import torch

# 1. Force close any previous instances to free up the port
gr.close_all()

def predict(message, history):
    # Helper to prevent data from breaking the Markdown table
    def sanitize(text):
        if text is None: return "N/A"
        return str(text).replace("|", " ").replace("\n", " ").strip()

    # 2. Search Databases
    query_vector = encoder.encode(message).tolist()
    
    try:
        nist_hits = qdrant.search(collection_name="nist_controls", query_vector=query_vector, limit=5)
        d3fend_hits = qdrant.search(collection_name="d3fend_techniques", query_vector=query_vector, limit=5)
        kev_hits = qdrant.search(collection_name="known_exploited_vulnerabilities", query_vector=query_vector, limit=5)
        attack_hits = qdrant.search(collection_name="mitre_attack", query_vector=query_vector, limit=5)
    except AttributeError:
        nist_hits = qdrant.query_points(collection_name="nist_controls", query=query_vector, limit=5).points
        d3fend_hits = qdrant.query_points(collection_name="d3fend_techniques", query=query_vector, limit=5).points
        kev_hits = qdrant.query_points(collection_name="known_exploited_vulnerabilities", query=query_vector, limit=5).points
        attack_hits = qdrant.query_points(collection_name="mitre_attack", query=query_vector, limit=5).points

    # 3. Prepare Markdown Tables
    # nist_md = "### Relevant NIST 800-53 Controls\n| ID | Title | Description Snippet | Score |\n| :--- | :--- | :--- | :--- |\n"
    for hit in nist_hits:
        id_val = sanitize(hit.payload.get('identifier'))
        title_val = sanitize(hit.payload.get('name'))
        desc_raw = hit.payload.get('control_text', '')
        desc_val = sanitize(desc_raw[:100]) + "..." if len(str(desc_raw)) > 100 else sanitize(desc_raw)
        
        # Ensure score is a float and formatted correctly
        score_val = f"{float(hit.score):.4f}" if hasattr(hit, 'score') and hit.score is not None else "0.0000"
        # nist_md += f"| {id_val} | {title_val} | {desc_val} | {score_val} |\n"

    # d3fend_md = "\n### Relevant MITRE D3FEND Techniques\n| ID | Technique | Description Snippet | Score |\n| :--- | :--- | :--- | :--- |\n"
    for hit in d3fend_hits:
        id_val = sanitize(hit.payload.get('ID'))
        tech_val = sanitize(hit.payload.get("D3FEND Technique Level 0") or hit.payload.get("Technique"))
        desc_raw = hit.payload.get('Definition', '')
        desc_val = sanitize(desc_raw[:100]) + "..." if len(str(desc_raw)) > 100 else sanitize(desc_raw)
        score_val = f"{float(hit.score):.4f}" if hasattr(hit, 'score') and hit.score is not None else "0.0000"
        # d3fend_md += f"| {id_val} | {tech_val} | {desc_val} | {score_val} |\n"

    # Header for Attack
    #attack_md = "### Relevant MITRE ATT&CK Controls\n| ID | Title | Description Snippet | Score |\n| :--- | :--- | :--- | :--- |\n"
    for hit in attack_hits:
        id_val = sanitize(hit.payload.get('ID'))
        title_val = sanitize(hit.payload.get('name'))
        desc_raw = hit.payload.get('description', '')
        desc_val = sanitize(desc_raw[:100]) + "..." if len(str(desc_raw)) > 100 else sanitize(desc_raw)
        score_val = f"{float(hit.score):.4f}" if hasattr(hit, 'score') and hit.score is not None else "0.0000"
        # attack_md += f"| {id_val} | {title_val} | {desc_val} | {score_val} |\n"

    # Header for Known Exploited Vulnerabilities
    #kev_md = "### Relevant Known Exploited Vulnerabilities \n| ID | Title | Description Snippet | Score |\n| :--- | :--- | :--- | :--- |\n"
    for hit in kev_hits:
        id_val = sanitize(hit.payload.get('cveID'))
        title_val = sanitize(hit.payload.get('vulnerabilitiyName'))
        desc_raw = hit.payload.get('shortDescription', '')
        desc_val = sanitize(desc_raw[:100]) + "..." if len(str(desc_raw)) > 100 else sanitize(desc_raw)
        score_val = f"{float(hit.score):.4f}" if hasattr(hit, 'score') and hit.score is not None else "0.0000"
        #kev_md += f"| {id_val} | {title_val} | {desc_val} | {score_val} |\n"

    
    # 4. Prepare Context for LLM
    nist_context = "\n".join([f"- ID: {hit.payload.get('identifier')} | Text: {hit.payload.get('control_text')}" for hit in nist_hits])
    d3fend_context = "\n".join([f"- ID: {hit.payload.get('ID')} | Text: {hit.payload.get('Definition')}" for hit in d3fend_hits])
    attack_context = "\n".join([f"- ID: {hit.payload.get('ID')} | Text: {hit.payload.get('description')}" for hit in attack_hits])
    kev_context = "\n".join([f"- ID: {hit.payload.get('cveID')} | Text: {hit.payload.get('shortDescription')}" for hit in kev_hits])
    combined_context = f"NIST Controls:\n{nist_context}\n\nD3FEND Techniques:\n{d3fend_context}\n\nATT&CK Techniques{attack_context}\n\nKnown Exploitable Vulnerabilities{kev_context}"

    #combined_context = f"NIST Controls:\n{nist_context}\n\nD3FEND Techniques:\n{d3fend_context}"
    
    # 4. Stream Answer
    system_prompt = """You are an expert Cybersecurity Advisor specializing in NIST SP 800-53 and MITRE D3FEND. Your goal is to provide cybersecurity knowledge to small or medium companies as well as individuals who aren't inherently security aware.

---
CORE OPERATING RULES:
1. RELEVANCE FILTER: Only include NIST controls that directly solve the user's problem.
2. SOURCE TRUTH: Use ONLY the provided NIST and D3FEND context.
3. NO HALLUCINATION: Do not invent control IDs or technical requirements.
4. TONE: Use plain, professional language.

---
RESPONSE STRUCTURE:
Issue Summary:
Explain the risk in plain language. Focus on the real-world consequence (e.g., "A hacker could steal your customer list if...") rather than just naming the threat.

Recommended Mitigations:
Provide specific, actionable steps a small team can take. Prioritize low-cost, high-impact actions (like MFA, backups, or training).

Relevant NIST Controls:
List only the most relevant controls. For each:
- [Control ID]: Provide a "Small Business Translation." Do not just copy the NIST snippet; explain specifically how this control protects the user's business in this scenario.

Relevant D3FEND Controls:
List only the most relevant controls. For each:
- [D3FEND ID]: Provide a "Small Business Translation." Do not just copy the D3FEND snippet; explain specifically how this control protects the user's business in this scenario.

Relevant KEV Entries:
List only the most relevant vulnerabilities. Reference CISA KEV if the vulnerability is known to be actively exploited.
For each:
- [CVE ID]: Provide a "Real-World Risk Translation." Explain if hackers are actively using this and why it matters for their specific software or setup.

Relevant ATT&CK Techniques:
List only the most relevant adversary methods. Reference MITRE ATT&CK to explain the adversary's offensive strategy.
For each:
- [Technique ID]: Provide a "Hacker Playbook Translation." Explain in simple terms how a hacker would use this specific technique to attack a business like theirs.

---
STRICT LIMITATIONS:
- Prioritize accuracy and practical utility over length. 
- Never reference external standards (ISO, CIS) unless they are in the context.
- Avoid repetitive mapping (e.g., don't list both a parent control and its enhancement unless both add unique value)."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Context:\n{combined_context}\n\nQuestion: {message}\n\nAnswer:"}
    ]

    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
    attention_mask = torch.ones_like(input_ids)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    generation_kwargs = dict(
        input_ids=input_ids, 
        attention_mask=attention_mask, 
        streamer=streamer, 
        max_new_tokens=1200
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # Final Output Construction
    # table_header = nist_md + d3fend_md + "\n---\n### Advisor Recommendation\n"
    
    generated_text = ""
    for new_text in streamer:
        generated_text += new_text
        # yield table_header + generated_text
        yield generated_text

# Launch the Gradio Interface
demo = gr.ChatInterface(
    fn=predict,
    title="Cybersecurity RAG Advisor",
    description="Sources (NIST/D3FEND) and expert recommendations are displayed below.",
    theme="soft",
    examples=["What is MFA?", "Securing remote work computers"],
    type="messages",
    flagging_options=["Like", "Spam", "Inappropriate", "Other"],
    save_history=True
)

if __name__ == "__main__":
    demo.launch(share=True)

'''

In [ ]:
import gradio as gr
from threading import Thread
import torch
from transformers import TextIteratorStreamer # Assuming this was imported elsewhere in your environment

# 1. Force close any previous instances to free up the port
gr.close_all()

def predict(message, history):
    # Helper to prevent data from breaking the Markdown table
    def sanitize(text):
        if text is None: return "N/A"
        return str(text).replace("|", " ").replace("\n", " ").strip()

    # 2. Search Databases
    query_vector = encoder.encode(message).tolist()
    
    try:
        nist_hits = qdrant.search(collection_name="nist_controls", query_vector=query_vector, limit=5)
        d3fend_hits = qdrant.search(collection_name="d3fend_techniques", query_vector=query_vector, limit=5)
        kev_hits = qdrant.search(collection_name="known_exploited_vulnerabilities", query_vector=query_vector, limit=5)
        attack_hits = qdrant.search(collection_name="mitre_attack", query_vector=query_vector, limit=5)
    except AttributeError:
        nist_hits = qdrant.query_points(collection_name="nist_controls", query=query_vector, limit=5).points
        d3fend_hits = qdrant.query_points(collection_name="d3fend_techniques", query=query_vector, limit=5).points
        kev_hits = qdrant.query_points(collection_name="known_exploited_vulnerabilities", query=query_vector, limit=5).points
        attack_hits = qdrant.query_points(collection_name="mitre_attack", query=query_vector, limit=5).points

    # 3. Prepare Context for LLM
    nist_context = "\n".join([f"- ID: {hit.payload.get('identifier')} | Text: {hit.payload.get('control_text')}" for hit in nist_hits])
    d3fend_context = "\n".join([f"- ID: {hit.payload.get('ID')} | Text: {hit.payload.get('Definition')}" for hit in d3fend_hits])
    attack_context = "\n".join([f"- ID: {hit.payload.get('ID')} | Text: {hit.payload.get('description')}" for hit in attack_hits])
    kev_context = "\n".join([f"- ID: {hit.payload.get('cveID')} | Text: {hit.payload.get('shortDescription')}" for hit in kev_hits])
    combined_context = f"NIST Controls:\n{nist_context}\n\nD3FEND Techniques:\n{d3fend_context}\n\nATT&CK Techniques{attack_context}\n\nKnown Exploitable Vulnerabilities{kev_context}"

    # 4. Stream Answer
    system_prompt = """You are an expert Cybersecurity Advisor specializing in NIST SP 800-53 and MITRE D3FEND. Your goal is to provide cybersecurity knowledge to small or medium companies as well as individuals who aren't inherently security aware.

---
CORE OPERATING RULES:
1. RELEVANCE FILTER: Only include NIST controls that directly solve the user's problem.
2. SOURCE TRUTH: Use ONLY the provided NIST and D3FEND context.
3. NO HALLUCINATION: Do not invent control IDs or technical requirements.
4. TONE: Use plain, professional language.

---
RESPONSE STRUCTURE:
Issue Summary:
Explain the risk in plain language. Focus on the real-world consequence (e.g., "A hacker could steal your customer list if...") rather than just naming the threat.

Recommended Mitigations:
Provide specific, actionable steps a small team can take. Prioritize low-cost, high-impact actions (like MFA, backups, or training).

Relevant NIST Controls:
List only the most relevant controls. For each:
- [Control ID]: Provide a "Small Business Translation." Do not just copy the NIST snippet; explain specifically how this control protects the user's business in this scenario.

Relevant D3FEND Controls:
List only the most relevant controls. For each:
- [D3FEND ID]: Provide a "Small Business Translation." Do not just copy the D3FEND snippet; explain specifically how this control protects the user's business in this scenario.

Relevant KEV Entries:
List only the most relevant vulnerabilities. Reference CISA KEV if the vulnerability is known to be actively exploited.
For each:
- [CVE ID]: Provide a "Real-World Risk Translation." Explain if hackers are actively using this and why it matters for their specific software or setup.

Relevant ATT&CK Techniques:
List only the most relevant adversary methods. Reference MITRE ATT&CK to explain the adversary's offensive strategy.
For each:
- [Technique ID]: Provide a "Hacker Playbook Translation." Explain in simple terms how a hacker would use this specific technique to attack a business like theirs.

---
STRICT LIMITATIONS:
- Prioritize accuracy and practical utility over length. 
- Never reference external standards (ISO, CIS) unless they are in the context.
- Avoid repetitive mapping (e.g., don't list both a parent control and its enhancement unless both add unique value)."""

    # --- NEW: HISTORY LOGIC STARTS HERE ---
    # Start with the system prompt
    messages = [{"role": "system", "content": system_prompt}]

    # Append previous chat history
    for entry in history:
        messages.append(entry)

    # Append the current user message with the RAG context
    # Note: Context is only added to the current query to keep the history clean
    messages.append({
        "role": "user", 
        "content": f"Context for the current query:\n{combined_context}\n\nQuestion: {message}"
    })
    # --- NEW: HISTORY LOGIC ENDS HERE ---

    # 5. Stream Answer
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
    attention_mask = torch.ones_like(input_ids)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    generation_kwargs = dict(
        input_ids=input_ids, 
        attention_mask=attention_mask, 
        streamer=streamer, 
        max_new_tokens=1200
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    generated_text = ""
    for new_text in streamer:
        generated_text += new_text
        yield generated_text

# Launch the Gradio Interface
demo = gr.ChatInterface(
    fn=predict,
    title="Cybersecurity RAG Advisor",
    description="Sources (NIST/D3FEND) and expert recommendations are displayed below.",
    theme="soft",
    examples=["What is MFA?", "Securing remote work computers"],
    type="messages", # This ensures history is passed as a list of dicts
    flagging_options=["Like", "Spam", "Inappropriate", "Other"],
    save_history=True
)

if __name__ == "__main__":
    demo.launch(share=True)